### Data Analysis 3 - Assignment 
#### 1. Data choice & wrangling

I used Airbnb listings data for Berlin obtained from Inside Airbnb. 
The main dataset contains more than 14,000 observations. 
An earlier quarterly snapshot of Berlin listings is used later for time-validity analysis. Later on I used Geneva for spatial validity.


In [11]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("../data/processed/berlin_2025_Q3_clean.csv")
df.shape

(9264, 13)

In [ ]:
# Select relevant columns
cols = [
    "price",
    "log_price",
    "accommodates",
    "bedrooms",
    "bathrooms",
    "room_type",
    "latitude",
    "longitude",
    "review_scores_rating",
    "number_of_reviews",
    "wifi",
    "kitchen",
    "air_conditioning"
]

df = df[cols]

In [14]:
# Clean and transform the price column

df["price"] = (
    df["price"]
    .replace('[\$,]', '', regex=True)
    .astype(float)
)

df = df[df["price"] > 0]
df["log_price"] = np.log(df["price"])


<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:5: SyntaxWarning: invalid escape sequence '\$'
C:\Users\admin\AppData\Local\Temp\ipykernel_13796\2206484955.py:5: SyntaxWarning: invalid escape sequence '\$'
  .replace('[\$,]', '', regex=True)


##### Airbnb prices are highly right-skewed; therefore, we model the logarithm of price to stabilize variance and improve model performance.

In [16]:
# Handle missing values
df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].median())
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())
df["review_scores_rating"] = df["review_scores_rating"].fillna(0)
df["number_of_reviews"] = df["number_of_reviews"].fillna(0)
df["room_type"] = df["room_type"].fillna("Unknown")

In [19]:
# Feature engineering on amenities
if "amenities" in df.columns:
    df["wifi"] = df["amenities"].str.contains("Wifi|Wi-Fi", case=False, na=False).astype(int)
    df["kitchen"] = df["amenities"].str.contains("Kitchen", case=False, na=False).astype(int)
    df["air_conditioning"] = df["amenities"].str.contains("Air conditioning", case=False, na=False).astype(int)

    df = df.drop(columns=["amenities"])


In [20]:
# Save the cleaned dataset
df.to_csv("../data/processed/berlin_2025_Q3_clean.csv", index=False)

Variables were selected based on economic intuition and prior Airbnb literature. 
Capacity-related variables (accommodates, bedrooms, bathrooms) capture scale effects, 
while location (latitude, longitude), amenities, and room type capture quality differences. 
Review scores proxy for reputation effects.
